In [5]:
# First I decided to use github again to get my downloaded data just so you can easily run this code yourself
# The excel sheet is downloaded as a file in the github repo but I'm not sure you can read it yourself so I will attach it in the Brightspace
!git clone https://github.com/CLBStyle/Digital_Methods_History-Final-Project.git
%cd Digital_Methods_History-Final-Project

Cloning into 'Digital_Methods_History-Final-Project'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 21 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 180.51 KiB | 18.05 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Digital_Methods_History-Final-Project/Digital_Methods_History-Final-Project/Digital_Methods_History-Final-Project/Digital_Methods_History-Final-Project


In [7]:
#My program was having some trouble finding the files to read because I kept renaming it
!ls data

immigration_data.xlsx  README.md


In [23]:
# The excel sheet was horrible and unreadable by pandas because the xml was essentially invalid.
# I tried to convert it to a csv but that didn't work either
# To actually open the file so you can create an editable data frame I found there was an engine "calamine" that can still export a data frame
# from messy data
import pandas as pd
!pip install python-calamine

# Now pandas can read the files and you can print the datatable.
df = pd.read_excel(
    "data/immigration_data.xlsx",
    engine="calamine",
    #I needed to skip rows because there was a bunch of junk rows on the excel that were not important for my figure
    skiprows=8
)

print(df.head())
print(df.columns)

  Region and Country of Birth      2000      2006       Unnamed: 3      2007  \
0                         NaN  Estimate  Estimate  Margin of Error  Estimate   
1         Total Foreign Born*  31107889  37547315           125604  38059555   
2                      Europe   4915557   4993135            48762   4990294   
3             Northern Europe    974619    953460            16118    939589   
4                     Denmark     30000         -                -         -   

        Unnamed: 5      2008       Unnamed: 7      2009       Unnamed: 9  ...  \
0  Margin of Error  Estimate  Margin of Error  Estimate  Margin of Error  ...   
1           119486  37960773           122987  38517104           115704  ...   
2            45508   4969090            42281   4887221            50207  ...   
3            15352    951374            16595    937715            18644  ...   
4                -         -                -     32966             4456  ...   

       2019      Unnamed: 29    

In [24]:
# Now there is a datatable that I can see wit the the real numbers that I will be using for my flow chart.
# However, I need to clean the datatable and organize it so that I can work with data
clean_columns = []

# This Loops through each column and checks if there is an integer in it, otherwise setting it aside
#gets rid of the columns like "unnamed" and "margin of error."
# Since the only key data includes the years,
# the regions and the corresponding data, all the other junk can be deleted.
for col in df.columns:
    if isinstance(col, (int, float)) or str(col).isdigit():
        clean_columns.append(col)

# This manually adds columns like the regions column and ensures that all valuable columns are saved
df_clean = df[[df.columns[0]] + clean_columns]

# Rename first column ONCE. Need to do this to simplify and organize my regions better.
df_clean = df_clean.rename(columns={df.columns[0]: "region"})

# This removes the "Estimate" row
df_clean = df_clean.drop(index=0).reset_index(drop=True)

# Replace dashes with missing values
df_clean = df_clean.replace("-", pd.NA)

# Convert year columns to numbers
year_cols = [col for col in df_clean.columns if col != "region"]

# This step converts all year columns into numeric values and replaces any non-numeric entries with missing values
#ensures that the dataset can be used for data analysis and visualization
for col in year_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Printing the data to see that it is clean and organized
print(df_clean.head())
print(df_clean.columns)

                region        2000        2006        2007        2008  \
0  Total Foreign Born*  31107889.0  37547315.0  38059555.0  37960773.0   
1               Europe   4915557.0   4993135.0   4990294.0   4969090.0   
2      Northern Europe    974619.0    953460.0    939589.0    951374.0   
3              Denmark     30000.0         NaN         NaN         NaN   
4              Ireland    156474.0    133433.0    135722.0    134897.0   

         2009        2010        2011        2012        2013        2014  \
0  38517104.0  39955673.0  40377757.0  40824553.0  41347945.0  42390705.0   
1   4887221.0   4817437.0   4889987.0   4809392.0   4803059.0   4764822.0   
2    937715.0    923564.0    946984.0    929873.0    952872.0    930580.0   
3     32966.0     29441.0     32171.0     26360.0     28521.0     26920.0   
4    121982.0    124457.0    132540.0    132277.0    128350.0    125022.0   

         2015        2016        2017        2018        2019        2021  \
0  43289646.0  

/tmp/ipykernel_5627/1650609027.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df_clean.replace("-", pd.NA)


In [25]:
# I learned that wide format is good for reading but very difficult to plot and animate. I started to learn that I had to pull the years out
# of the columns manually and that plotly does not function well with wide formating columns.
# Long format columns automate everything and streamline the process so that you don't have to update your column logic everywhere.
df_long = df_clean.melt(
    id_vars="region",
    var_name="year",
    value_name="count"
)

# Remove empty values
df_long = df_long.dropna(subset=["count"])

# Make year numeric; prevents the columns from interpreting the numbers as strings
df_long["year"] = df_long["year"].astype(int)

print(df_long.head())

                region  year       count
0  Total Foreign Born*  2000  31107889.0
1               Europe  2000   4915557.0
2      Northern Europe  2000    974619.0
3              Denmark  2000     30000.0
4              Ireland  2000    156474.0


In [26]:
import plotly.graph_objects as go
# The dataset had a ton of countries but I realized it would take my already long project way too long. I decided to focus on continents
# for the regions. First you must name and sort them
regions_to_map = ["Europe", "Asia", "Latin America", "Africa"]

# Dataset has regions already written in; this will sort through the columns and look for the numbers of these particular regions
map_df = df_long[df_long["region"].isin(regions_to_map)].copy()

#Only prints the first 5 columns but now we can see we have a plotable dataset
print(map_df.head())
print(map_df["region"].unique())

            region  year       count
1           Europe  2000   4915557.0
43            Asia  2000   8226254.0
92          Africa  2000    881300.0
136  Latin America  2000  16086974.0
194         Europe  2006   4993135.0
['Europe' 'Asia' 'Africa' 'Latin America']


In [27]:
# You actually have to find the coordinates to make the graphic work. I asked ChatGPT for the ideal points on the globe to place for each region
region_coords = {
    "Europe": {"lat": 54.5, "lon": 15.2},
    "Asia": {"lat": 34.0, "lon": 100.0},
    "Latin America": {"lat": -10.0, "lon": -60.0},
    "Africa": {"lat": 1.5, "lon": 17.0}
}
# This function converts the regional labels into readable map positions
map_df["lat"] = map_df["region"].map(lambda x: region_coords[x]["lat"])
map_df["lon"] = map_df["region"].map(lambda x: region_coords[x]["lon"])

In [28]:
usa_lat = 37.0902
usa_lon = -95.7129

# This finds all the unique years in the dataset and sorts them in order, finding the largest number in the data set as a reference point
# For our calculation to get line width
years = sorted(map_df["year"].unique())
max_count = map_df["count"].max()

# This creates the line width that will show immigration flows, dividing the count by the max count and multiplying by a constant for better
# visibility
def line_width(count):
    return (count / max_count) * 12

In [29]:
# Creates the map
fig = go.Figure()
# Picks first year
first_year = years[0]
# Filters data for first year
first_df = map_df[map_df["year"] == first_year]

# This giant for loop sets the basis for the visual. It loops through each region and draws lines using scattergeo. These lines are
# connected to the lattitude and longitude points we set for the regions, all connecting to the USA.
for _, row in first_df.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row["lon"], usa_lon],
        lat=[row["lat"], usa_lat],
        mode="lines",
        # This part sets the line thickness, allowing it to change in terms of the number. One challenge includes choosing a line thickness
        # that can exhibit visible change to the audiance while also not being too thick so as to cover the map.
        line=dict(width=line_width(row["count"])),
        opacity=0.7,
        # Legend naming
        name=row["region"],
        #I asked chat how to create a hover visual over the graph so you can see the actual immigration numbers as the line thickness changes.
        # It helped me create the shortcut to build this part of the project. Note: you can only see the hover visual over the origin point of
        # the line for each region if you want the yearly immigration numbers.
        hoverinfo="text",
        text=f'{row["region"]}<br>{row["year"]}<br>{int(row["count"]):,}'
    ))

In [30]:
# This cell converst the static map to an animation that will show the changing line thickness over time
frames = []
# Loops through each year and filters data for each year
for year in years:
    year_df = map_df[map_df["year"] == year]
   # This will hold the lines all for one year
   frame_data = []
# Loops through each region in that year
    for _, row in year_df.iterrows():
      # The same code but not it's looped for animation
      # Frame_data.append stores the drawing instructions I gave previously and then can iterate over it
        frame_data.append(go.Scattergeo(
            lon=[row["lon"], usa_lon],
            lat=[row["lat"], usa_lat],
            mode="lines",
            line=dict(width=line_width(row["count"])),
            opacity=0.7,
            name=row["region"],
            hoverinfo="text",
            text=f'{row["region"]}<br>{row["year"]}<br>{int(row["count"]):,}'
        ))
# This stores the frame so it can create all the lines from every region for each year
    frames.append(go.Frame(data=frame_data, name=str(year)))
# This tells plotly what all the images needed for the animation
fig.frames = frames

In [31]:
# Now we have the animation and just have to project it correctly according to the settings of plotly. I had no idea how to do this.
# I looked up plotly functions and asked ChatGPT for help on how to set sliders and the map.
fig.update_layout(
    #This section handles creating the legend and the graphics from which you can see the animation. For example, you can easily customize
    #your animation to portray the globe.
    title=f"Foreign-Born Population in the U.S. by Region of Birth ({first_year})",
    geo=dict(
        scope="world",
        projection_type="natural earth",
        showland=True
    ),
    # This part handles animation control, I needed a lot of help in setting this up. You can customize the duration of the animation for each
    # image change.
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {
                "frame": {"duration": 700, "redraw": True},
                "fromcurrent": True
            }]
        }]
    }],
    #This is so you can scroll through the data yourself with a slider in addition to being able to play the simulation on the screen
    # It essentially uses the same code as above
    sliders=[{
        "steps": [{
            "label": str(year),
            "method": "animate",
            #Tells plotly to only focus on that one year when someone clicks on it
            "args": [[str(year)], {
                "frame": {"duration": 300, "redraw": True},
                # Wheras the animation code above fades the plot changes, the sliders transition immediately
                "mode": "immediate"
            }]
        } for year in years]
    }]
)

fig.show()
# NOTE: the purple line represents all of Latin America including countries in Central and North America that are considered Hispanic.
# Asia represents the whole of Asia including the Middle East